# Extract Boundary Tokens WITH OFFSETS
Extrait les boundary tokens ET les positions caractères exactes

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"Working dir: {os.getcwd()}")

In [ ]:
!pip install transformers torch tqdm -q
print("OK")

In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification
from tqdm import tqdm

# Load model
print("[1] Load model...")
model_path = Path('checkpoints/camelbert_binary_classification_final')
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForTokenClassification.from_pretrained(str(model_path))
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
print("OK")

In [ ]:
# Load corpus
print("\n[2] Load corpus...")
with open('data/processed/kitab_uqala_reference_corpus.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(f"      {len(text):,} chars")

In [ ]:
# Process FULL TEXT in chunks and keep track of offsets
print("\n[3] Process full corpus...")

all_tokens = []
all_predictions = []
all_offsets = []

CHUNK_SIZE = 500

for start_char in tqdm(range(0, len(text), CHUNK_SIZE), desc="Chunks"):
    end_char = min(start_char + CHUNK_SIZE + 50, len(text))
    chunk_text = text[start_char:end_char]
    
    # Encode WITH offset mapping
    encoded = tokenizer(
        chunk_text,
        return_tensors='pt',
        return_offsets_mapping=True,
        truncation=False,
        padding=False,
    )
    
    # Inference
    with torch.no_grad():
        if torch.cuda.is_available():
            outputs = model(
                input_ids=encoded['input_ids'].cuda(),
                attention_mask=encoded['attention_mask'].cuda()
            )
        else:
            outputs = model(**encoded)
        logits = outputs.logits[0]
    
    # Get predictions + offsets
    preds = np.argmax(logits.cpu().numpy(), axis=-1)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
    offsets = encoded['offset_mapping'][0].numpy()
    
    # Skip first 5 tokens of new chunks to avoid duplicates
    if len(all_tokens) > 0:
        preds = preds[5:]
        tokens = tokens[5:]
        offsets = offsets[5:]
    
    # Adjust offsets to absolute positions
    adjusted_offsets = []
    for char_start, char_end in offsets:
        adjusted_offsets.append((char_start + start_char, char_end + start_char))
    
    all_tokens.extend(tokens)
    all_predictions.extend(preds.tolist())
    all_offsets.extend(adjusted_offsets)

print(f"\nProcessing complete")
print(f"  Total tokens: {len(all_tokens):,}")
print(f"  Boundary tokens: {sum(all_predictions):,}")

In [ ]:
# Extract boundary tokens with their offsets
print("\nExtracting boundary tokens with offsets...")

boundary_tokens = []
boundary_offsets = []
boundary_indices = []

for idx, (token, pred, offset) in enumerate(zip(all_tokens, all_predictions, all_offsets)):
    if pred == 1:  # Boundary
        boundary_tokens.append(token)
        boundary_indices.append(idx)
        boundary_offsets.append({
            'char_start': int(offset[0]),
            'char_end': int(offset[1]),
            'token': token
        })

print(f"Extracted: {len(boundary_tokens):,} boundary tokens")
print(f"\nFirst 10 boundary token offsets:")
for i, off in enumerate(boundary_offsets[:10]):
    char_start = off['char_start']
    char_end = off['char_end']
    actual_text = text[char_start:char_end]
    print(f"  {i}: [{char_start:6d}:{char_end:6d}] '{actual_text[:30]}' (token: {off['token']})")

In [ ]:
# Save WITH offsets
print("\nSaving...")

results = {
    'metadata': {
        'corpus': 'kitab_uqala_reference_corpus.txt',
        'corpus_size_chars': len(text),
        'corpus_size_tokens': len(all_tokens),
        'model': 'camelbert_binary_classification_final',
        'boundary_tokens_count': len(boundary_tokens),
        'boundary_percentage': round(100 * len(boundary_tokens) / len(all_tokens), 2),
    },
    'boundary_tokens': boundary_tokens,
    'boundary_indices': boundary_indices,
    'boundary_offsets': boundary_offsets,  # NEW: Character positions
}

with open('results/camelbert_boundary_tokens_with_offsets.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✓ Saved: results/camelbert_boundary_tokens_with_offsets.json")
print(f"\nStats:")
print(f"  Boundary tokens: {len(boundary_tokens):,}")
print(f"  Percentage: {results['metadata']['boundary_percentage']}%")